# Generación de archivos './processed' (inputs DG)

Versiones de las librerías utilizadas

A continuación se detallan las versiones de las librerías empleadas en el presente código. Las pertenecientes a la biblioteca estándar se indican con “(stdlib)” y muestran la versión de Python del entorno.

| package          | version             |
| ---------------- | ------------------- |
| numpy            | 1.24.4              |
| pandas           | 2.0.3               |
| geopandas        | 0.13.2              |
| shapely          | 2.0.7               |
| matplotlib       | 3.7.5               |
| scipy            | 1.10.1              |
| pyproj           | 3.5.0 (PROJ 9.2.0)  |
| fiona            | 1.10.1 (GDAL 3.9.1) |
| pyogrio          | 0.9.0               |
| rtree            | 1.3.0               |
| os (stdlib)      | Python 3.8.10       |
| pathlib (stdlib) | Python 3.8.10       |

*Backends nativos detectados: GEOS 3.11.4; PROJ 9.2.0; GDAL 3.9.1.*

Chunk 0: imports + paths + cargar capas base

In [ ]:
# Chunk 0 — Configuración, imports y capas base
#
# Este cuaderno construye los objetos serializados que consume Deep Gravity
# y, en su segunda mitad, audita los resultados de la corrida.
#
# Chunks 0–4: operan sobre infraestructura territorial (Nivel 1) y son
#             ejecutables con lo que el repositorio distribuye.
# Chunk 5 en adelante: dependen de flows.csv y de la corrida del modelo,
#             ninguno de los cuales forma parte de la entrega pública.

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import config as cfg

import json
import pickle
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from shapely.geometry import LineString
from matplotlib.patches import Patch
from matplotlib.colors import Normalize, LogNorm, TwoSlopeNorm
from matplotlib.cm import ScalarMappable, get_cmap
from matplotlib.ticker import FuncFormatter
from math import radians, sin, cos, asin

# Rutas del repositorio
BASE = cfg.CHILE_DIR
PROCESSED = cfg.PROCESSED_DIR
DG_ROOT = cfg.DG_ROOT
RES = cfg.RESULTS_DIR
IMGS = cfg.IMGS_DIR

# Cargar Output Areas y teselación
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(cfg.CRS_LATLON)

try:
    tiles = gpd.read_file(cfg.TESSELLATION_GEOJSON, engine="pyogrio")
except Exception:
    tiles = gpd.read_file(cfg.TESSELLATION_SHP)
if tiles.crs is None or tiles.crs.to_epsg() != 4326:
    tiles = tiles.to_crs(cfg.CRS_LATLON)

assert "OA_ID" in oas.columns, "OA_ID no existe en output_areas"
assert "tile_id" in tiles.columns, "tile_id no existe en tessellation"

len(oas), len(tiles)

Chunk 1 — oa2centroid.pkl y oa_gdf.csv.gz

In [ ]:
# Chunk 1 — oa2centroid.pkl y oa_gdf.csv.gz
# --- punto representativo para centroides robustos dentro del polígono ---
oa_pts = oas[["OA_ID", "geometry"]].copy()
oa_pts["geometry"] = oa_pts.geometry.representative_point()

# Dict {OA_ID: [lat, lon]} en grados (EPSG:4326)
oa2centroid = {str(row.OA_ID): [row.geometry.y, row.geometry.x] for _, row in oa_pts.iterrows()}
with open(PROCESSED / "oa2centroid.pkl", "wb") as fh:
    pickle.dump(oa2centroid, fh)

# oa_gdf.csv.gz (al menos el ID; duplicamos como geo_code por compatibilidad)
oa_gdf = pd.DataFrame({"OA_ID": oas["OA_ID"].astype(str)})
oa_gdf["geo_code"] = oa_gdf["OA_ID"]
oa_gdf.to_csv(PROCESSED / "oa_gdf.csv.gz", index=False, compression="gzip")

print("✔ oa2centroid.pkl y oa_gdf.csv.gz creados")
print("  Ejemplo centroid:", list(oa2centroid.items())[:1])

Chunk 2 — tileid2oa2handmade_features.json

In [ ]:
# Chunk 2 --- OA->tile y JSON tileid2oa2handmade_features ---
# Usa oa_pts (del chunk 1) y tiles (del chunk 0) ya cargados

join = gpd.sjoin(
    oa_pts[["OA_ID", "geometry"]],
    tiles[["tile_id", "geometry"]],
    how="left",
    predicate="within",
)

# QA: toda OA debe tener tile
assert join["tile_id"].notna().all(), "Hay OA sin tile; revisar CRS/cobertura de la grilla"

# Dict {tile_id: {OA_ID: {}}}
tileid2oa = {
    str(tid): {str(oa): {} for oa in g["OA_ID"].astype(str).tolist()}
    for tid, g in join.groupby("tile_id")
}

# Guardar JSON
out_json = PROCESSED / "tileid2oa2handmade_features.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump(tileid2oa, fh, ensure_ascii=False)

tile_counts = join.groupby("tile_id").size().sort_values(ascending=False)
print(f"JSON creado: {out_json.name}")
print(f"Tiles con OA: {tile_counts.size} | OA total: {tile_counts.sum()}")
print("Top tiles por #OA:")
print(tile_counts.head(7))

Chunk 3 — Split train/test

In [ ]:
# Chunk 3 — Split train/test
# --- Split train/test determinístico (sin header), usando solo tiles con ≥2 OAs ---
tiles_sorted_all = tile_counts.index.tolist()
counts = tile_counts.to_dict()

# Filtra tiles con al menos 2 OAs (descarta los de 1 OA que no aportan)
tiles_ge2 = [tid for tid in tiles_sorted_all if counts[tid] >= 2]

# Ordena por número de OAs (desc), luego por id para estabilidad
tiles_ge2_sorted = sorted(tiles_ge2, key=lambda t: (-counts[t], t))
assert len(tiles_ge2_sorted) >= 2, "Se requieren ≥2 tiles con al menos 2 OAs para el split."

# TEST = 1° más grande por #OAs; TRAIN = el resto
test_tiles  = [tiles_ge2_sorted[0]]
train_tiles = [t for t in tiles_ge2_sorted if t not in test_tiles]

# Guardar SIN header (1 id por línea)
pd.Series(train_tiles, dtype=str).to_csv(PROCESSED / "train_tiles.csv", index=False, header=False)
pd.Series(test_tiles,  dtype=str).to_csv(PROCESSED / "test_tiles.csv",  index=False, header=False)

print("train/test tiles guardados en processed/")
print(f"  train={len(train_tiles)} | test={len(test_tiles)}")
print("  train tiles:", train_tiles)
print("  test tiles:", test_tiles)

Chunk 4 — oa2features.pkl (+ columnas usadas)

In [ ]:
# Chunk 4 — oa2features.pkl (+ columnas usadas)
# Leer features.csv
feat_path = BASE / "features.csv"
f = pd.read_csv(feat_path)

# QA básica
assert "OA_ID" in f.columns, "features.csv debe tener OA_ID"
assert set(f["OA_ID"].astype(str)) == set(oas["OA_ID"].astype(str)), "OA_ID en features.csv no calzan con output_areas"

# Columnas a excluir explícitamente (IDs, población/densidad/área y no-numéricas conocidas)
drop_explicit = {
    "OA_ID", "geo_code", "area_km2", "AREA_KM2",
    "population", "pop", "pob_total", "poblacion", "población",
    "pop_density", "density", "densidad",
    "tile_id", "REGION", "PROVINCIA", "COMUNA", "region", "provincia", "comuna",
    "geometry"
}

# Quedarnos con columnas numéricas válidas que NO estén en drop_explicit
num_cols = [c for c in f.columns if c not in drop_explicit and pd.api.types.is_numeric_dtype(f[c])]
use_cols = sorted(num_cols)  # orden estable

# Reemplazos defensivos
X = f[use_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype("float64")

# Construir dict {OA_ID: [f1...fK]}
oa_ids = f["OA_ID"].astype(str).tolist()
oa2features = {oa: X.iloc[i].tolist() for i, oa in enumerate(oa_ids)}

# Guardar
with open(PROCESSED / "oa2features.pkl", "wb") as fh:
    pickle.dump(oa2features, fh)

# Guardar también el orden de columnas para reproducibilidad
with open(PROCESSED / "oa2features_columns.json", "w", encoding="utf-8") as fh:
    json.dump(use_cols, fh, ensure_ascii=False, indent=2)

# Resumen
print("✔ oa2features.pkl creado")
print(f"  OAs: {len(oa2features)} | K (features por OA): {len(use_cols)}")
print("  Primeras columnas:", use_cols[:5])

---

### Dependencia del procedimiento telco

Desde este punto, el cuaderno consume `flows.csv`, producto del cuaderno
`preprocesamiento_trazas_telco`. Ese archivo deriva de la fuente de telefonía
móvil y, en cumplimiento de las restricciones de uso declaradas en el
manuscrito, no forma parte de esta entrega. Se regenera ejecutando dicho
cuaderno sobre una fuente de estructura equivalente.

Los Chunks 0 a 4 son ejecutables con lo que el repositorio distribuye.

Chunk 5 — flows_oa.csv.zip y od2flow.pkl (desde tu flows.csv)

In [ ]:
# Chunk 5 — flows_oa.csv.zip y od2flow.pkl (desde flows.csv)
flows_path = cfg.require(
    cfg.FLOWS_CSV,
    "el archivo flows.csv",
    nivel=2,
    detalle=(
        "No es una fuente externa, sino un producto del cuaderno "
        "preprocesamiento_trazas_telco.ipynb. Deriva de la fuente de "
        "telefonía móvil y no forma parte de esta entrega. Ejecute dicho "
        "cuaderno sobre su propia fuente para generarlo."
    ),
)
flows = pd.read_csv(flows_path)

# Normalizar nombres de columnas a minúsculas y validar
flows.columns = [c.lower() for c in flows.columns]
assert {"origin", "destination", "flow"}.issubset(flows.columns), "flows.csv debe tener origin,destination,flow"

# Tipos y limpieza básica
flows["origin"] = flows["origin"].astype(str)
flows["destination"] = flows["destination"].astype(str)
flows["flow"] = flows["flow"].astype(float)

# Filtrar a OA válidas (por si hubiera IDs fuera de output_areas)
valid_ids = set(oa_gdf["OA_ID"].astype(str))
flows = flows[flows["origin"].isin(valid_ids) & flows["destination"].isin(valid_ids)].copy()

# Agregar por si hay filas duplicadas del mismo par OD
flows = flows.groupby(["origin", "destination"], as_index=False)["flow"].sum()

# Mantener o no la diagonal:
# (evaluate() detectará automáticamente si hay diagonal y actuará en consecuencia)
has_diag = (flows["origin"] == flows["destination"]).any()
diag_count = int((flows["origin"] == flows["destination"]).sum())
diag_flow_sum = float(flows.loc[flows["origin"] == flows["destination"], "flow"].sum())

# Convertir a int si corresponde (tras agregación pueden quedar floats por lectura)
flows["flow"] = flows["flow"].round().astype(int)

# Guardar en formato esperado por DG
flows_oa = flows.rename(
    columns={"origin": "residence", "destination": "workplace", "flow": "commuters"}
)
flows_oa.to_csv(PROCESSED / "flows_oa.csv.zip", index=False, compression="zip")

# Diccionario {(o,d): flow}
od2flow = {(o, d): int(v) for o, d, v in flows_oa[["residence", "workplace", "commuters"]].itertuples(index=False)}
with open(PROCESSED / "od2flow.pkl", "wb") as fh:
    pickle.dump(od2flow, fh)

# Resumen de control
O = flows_oa.groupby("residence")["commuters"].sum()
D = flows_oa.groupby("workplace")["commuters"].sum()
print("flows_oa.csv.zip y od2flow.pkl creados")
print(f"  OD pares: {len(flows_oa)} | O>0: {(O>0).sum()} OAs | D>0: {(D>0).sum()} OAs")
print(f"  Diagonal presente: {has_diag} | filas diagonal: {diag_count} | flujo diagonal total: {diag_flow_sum:.0f}")

Finalmente, generamos la carpeta ''.\results' para ir guardando los outputs de nuestro modelo

In [ ]:
# Directorio de salidas del modelo
RES.mkdir(parents=True, exist_ok=True)
print("Directorio de resultados listo: deepgravity/results/")

---

### Dependencia de la corrida del modelo

Las secciones siguientes consumen `results/edges_TEST_pairs.csv` y
`results/tile2cpc_DG_chile.csv`, generados por `main.py`. No se distribuyen:
derivan de la variable objetivo construida a partir de la fuente telco.

Los valores de referencia de la corrida documentada figuran en el §12 del
informe de montaje, con carácter de registro documental y no de control
verificable por terceros.

# Visualizacion de resultados

In [ ]:
# --- [GRÁFICO #1] Mapa: Provincia de Santiago + comunas en tiles TEST ---

# Cargar comunas (OAs)
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

# Cargar mapping tile->OAs y lista de tiles TEST
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tile2oas = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()

# OAs incluidas en tiles de TEST (robusto si faltara algún tile en el JSON)
oas_test = set()
for tid in test_tiles:
    oas_test |= set(map(str, tile2oas.get(tid, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# Teselación para dibujar contorno de tiles TEST
try:
    tiles = gpd.read_file(cfg.TESSELLATION_GEOJSON, engine="pyogrio")
except Exception:
    tiles = gpd.read_file(cfg.TESSELLATION_SHP)
if tiles.crs is None or tiles.crs.to_epsg() != 4326:
    tiles = tiles.to_crs(4326)
tiles["tile_id"] = tiles["tile_id"].astype(str)
tiles_test = tiles[tiles["tile_id"].isin(test_tiles)].copy()

# Plot
fig, ax = plt.subplots(figsize=(12, 12))
oas.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#9e9e9e", linewidth=0.6, zorder=1, label="Comunas (todas)")
oas[oas["is_test"]].plot(ax=ax, facecolor="#ffb74d", edgecolor="#f57c00", linewidth=1.2, zorder=2, label="Comunas en tiles TEST")

if not tiles_test.empty:
    tiles_test.boundary.plot(ax=ax, color="#f57c00", linewidth=1.2, zorder=3)
    # Etiquetas de ID de tile (en punto representativo)
    tiles_test["__pt"] = tiles_test.representative_point()
    for _, r in tiles_test.iterrows():
        ax.text(r["__pt"].x, r["__pt"].y, r["tile_id"], fontsize=9, ha="center", va="center")

# Leyenda y formato
legend_elements = [
    Patch(facecolor="#f0f0f0", edgecolor="#9e9e9e", label="Comunas (todas)"),
    Patch(facecolor="#ffb74d", edgecolor="#f57c00", label="Comunas en tiles TEST"),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False)
title_tiles = ", ".join(test_tiles) if len(test_tiles) > 0 else "N/A"
ax.set_title(f"Provincia de Santiago — Comunas en tiles TEST: {title_tiles}")
ax.set_axis_off()
plt.tight_layout()

# Salidas: PNG alta resolución + PDF vectorial
png_path = IMGS / "map_test_communes.png"
pdf_path = IMGS / "map_test_communes.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()
plt.close(fig)

print("Guardados:", png_path.name, "|", pdf_path.name)

In [ ]:
# --- [GRÁFICO #2] Outflow intra-tile Oᵢ con barra de color ---

# Capas base
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

# Flujos observados (OD)
flows_oa = pd.read_csv(PROCESSED / "flows_oa.csv.zip")
flows_oa["residence"] = flows_oa["residence"].astype(str)
flows_oa["workplace"] = flows_oa["workplace"].astype(str)
flows_oa["commuters"] = flows_oa["commuters"].astype(int)

# Mapping tile -> OAs y tiles de TEST
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()

oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = {oa for tid in test_tiles if tid in tileid2oa for oa in tileid2oa[tid].keys()}

# Outflow intra-tile: solo i->j donde tile(i)==tile(j) y el ORIGEN está en tiles TEST
flows_oa = flows_oa[
    flows_oa["residence"].isin(oas_test) &
    flows_oa["workplace"].isin(oa2tile.keys())
].copy()

res_tile = flows_oa["residence"].map(oa2tile)
wrk_tile = flows_oa["workplace"].map(oa2tile)
flows_intra_test = flows_oa[res_tile == wrk_tile].copy()

if len(flows_oa):
    share_intra = flows_intra_test["commuters"].sum() / flows_oa["commuters"].sum()
    print(f"Proporción intra-tile (sobre total de orígenes TEST): {share_intra:.3f}")

# Outflow intra-tile Oᵢ y merge al mapa
O_intra = flows_intra_test.groupby("residence")["commuters"].sum().rename("O_intra").reset_index()
oas["is_test"] = oas["OA_ID"].isin(oas_test)
oas_O = oas.merge(O_intra, left_on="OA_ID", right_on="residence", how="left").fillna({"O_intra": 0})

# Choropleth + barra de color
fig, ax = plt.subplots(figsize=(12, 12))
oas.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#9e9e9e", linewidth=0.6, zorder=1)

oas_test_gdf = oas_O[oas_O["is_test"]].copy()
if not oas_test_gdf.empty:
    vmin = float(oas_test_gdf["O_intra"].min())
    vmax = float(oas_test_gdf["O_intra"].max())
    if vmin == vmax:
        vmax = vmin + 1.0
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap()
    oas_test_gdf.plot(column="O_intra", ax=ax, legend=False, cmap=cmap, vmin=vmin, vmax=vmax, zorder=2)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.034, pad=0.02)
    cbar.ax.set_title("Oᵢ intra-tile\n(Commuters)", fontsize=9)

oas.boundary.plot(ax=ax, linewidth=0.6, color="#9e9e9e", zorder=3)
ax.set_title("DeepGravity Chile — Outflow intra-tile Oᵢ por comuna (tiles TEST)")
ax.set_axis_off()
plt.tight_layout()

png_path = IMGS / "outflow_intra_choropleth_legend.png"
pdf_path = IMGS / "outflow_intra_choropleth_legend.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")

print("Guardados:", png_path.name, "|", pdf_path.name)

In [ ]:
# --- [GRÁFICO #3.1] Flujos observados (intra-tile, título incluye umbral) ---

SCALE_PATH = IMGS / "yobs_color_scale.json"   # escala compartida

# Estilos
ALL_FACE, ALL_EDGE = "#f0f0f0", "#9e9e9e"
TEST_FACE, TEST_EDGE = "#ffd1dc", "#f48fb1"

# OAs
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

# Tiles de TEST
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}

oas_test = set()
for t in test_tiles:
    oas_test |= set(map(str, tileid2oa.get(t, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# Edges (archivo generado por evaluate())
edges_path = RES / "edges_TEST_pairs.csv"
edges = pd.read_csv(edges_path)
edges.columns = [c.strip() for c in edges.columns]
need_cols = {"origin", "destination", "y_obs"}
assert need_cols.issubset(edges.columns), f"{edges_path.name} debe tener columnas {need_cols}"

# Coordenadas desde oa2centroid.pkl (formato [lat, lon])
if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].astype(str).map(get_lon)
    edges["lat_o"] = edges["origin"].astype(str).map(get_lat)
    edges["lon_d"] = edges["destination"].astype(str).map(get_lon)
    edges["lat_d"] = edges["destination"].astype(str).map(get_lat)

# Filtrar intra-tile sobre el conjunto TEST
edges = edges[edges["origin"].astype(str).isin(oas_test)]
edges = edges[edges["destination"].astype(str).isin(oas_test)]
same_tile = edges["origin"].astype(str).map(oa2tile) == edges["destination"].astype(str).map(oa2tile)
edges = edges[same_tile]
edges = edges[edges["y_obs"] > 0]
edges = edges.dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"]).copy()

# Escala compartida (cálculo o lectura)
if SCALE_PATH.exists():
    with open(SCALE_PATH, "r", encoding="utf-8") as fh:
        s = json.load(fh)
    vmin_all, vmax_all = float(s["vmin"]), float(s["vmax"])
else:
    vmin_all = float(edges["y_obs"].min()) if len(edges) else 0.0
    vmax_all = float(max(vmin_all + 1.0, edges["y_obs"].quantile(0.99))) if len(edges) else 1.0
    with open(SCALE_PATH, "w", encoding="utf-8") as fh:
        json.dump({"vmin": vmin_all, "vmax": vmax_all}, fh)

# Umbral del gráfico (prune)
PRUNE_Q = 0.75
thr = edges["y_obs"].quantile(PRUNE_Q) if len(edges) else 0.0
edges_plot = edges[edges["y_obs"] >= thr].copy()
if edges_plot.empty and len(edges):
    edges_plot = edges.sort_values("y_obs", ascending=False).head(50).copy()
    thr = float(edges_plot["y_obs"].min())

def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

g_edges = gpd.GeoDataFrame(edges_plot, geometry=edges_plot.apply(to_line, axis=1), crs="EPSG:4326")

# Plot
fig, ax = plt.subplots(figsize=(10, 9))
oas[~oas["is_test"]].plot(ax=ax, facecolor=ALL_FACE, edgecolor=ALL_EDGE, linewidth=0.6, zorder=1)
oas[oas["is_test"]].plot(ax=ax, facecolor=TEST_FACE, edgecolor=TEST_EDGE, linewidth=1.0, zorder=2)

if len(g_edges):
    norm = Normalize(vmin=vmin_all, vmax=vmax_all)
    cmap = get_cmap()
    g_plot = g_edges.sort_values("y_obs", ascending=True).reset_index(drop=True)
    vals = g_plot["y_obs"].clip(vmin_all, vmax_all).to_numpy()
    widths = 0.2 + 2.8 * norm(vals)
    colors = [cmap(norm(v)) for v in vals]
    for (_, r), lw, col in zip(g_plot.iterrows(), widths, colors):
        ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
                color=col, linewidth=lw, alpha=0.85, zorder=3)
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.set_title("Commuters\n(observado)", fontsize=9)

legend_elements = [
    Patch(facecolor=ALL_FACE, edgecolor=ALL_EDGE, label="Comunas (todas)"),
    Patch(facecolor=TEST_FACE, edgecolor=TEST_EDGE, label="Comunas en tiles TEST"),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False)
ax.set_title(
    f"Flujos observados — Provincia de Santiago (tiles TEST: {', '.join(test_tiles)})\n"
    f"Umbral p{int(PRUNE_Q*100)}: ≥ {thr:,.0f} commuters"
)
ax.set_axis_off()
plt.tight_layout()

png_path = IMGS / "observed_flows_TEST.png"
pdf_path = IMGS / "observed_flows_TEST.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print(f"[3.1] Escala compartida vmin={vmin_all:.0f}, vmax={vmax_all:.0f} | líneas={len(g_edges)} | umbral={thr:,.0f}")

In [ ]:
# --- [GRÁFICO #3.2] Flujos observados (intra-tile, TODOS los flujos) ---

SCALE_PATH = IMGS / "yobs_color_scale.json"   # el mismo archivo

ALL_FACE, ALL_EDGE = "#f0f0f0", "#9e9e9e"
TEST_FACE, TEST_EDGE = "#ffd1dc", "#f48fb1"

# Comunas y tiles
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}

oas_test = set()
for t in test_tiles:
    oas_test |= set(map(str, tileid2oa.get(t, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# Edges
edges_path = RES / "edges_TEST_pairs.csv"
edges = pd.read_csv(edges_path)
edges.columns = [c.strip() for c in edges.columns]
need_cols = {"origin", "destination", "y_obs"}
assert need_cols.issubset(edges.columns), f"{edges_path.name} debe tener columnas {need_cols}"

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].astype(str).map(get_lon)
    edges["lat_o"] = edges["origin"].astype(str).map(get_lat)
    edges["lon_d"] = edges["destination"].astype(str).map(get_lon)
    edges["lat_d"] = edges["destination"].astype(str).map(get_lat)

# Filtro intra-tile
edges = edges[edges["origin"].astype(str).isin(oas_test)]
edges = edges[edges["destination"].astype(str).isin(oas_test)]
same_tile = edges["origin"].astype(str).map(oa2tile) == edges["destination"].astype(str).map(oa2tile)
edges = edges[same_tile]
edges = edges[edges["y_obs"] > 0]
edges = edges.dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"]).copy()

# Escala compartida
if SCALE_PATH.exists():
    with open(SCALE_PATH, "r", encoding="utf-8") as fh:
        s = json.load(fh)
    vmin_all, vmax_all = float(s["vmin"]), float(s["vmax"])
else:
    vmin_all = float(edges["y_obs"].min()) if len(edges) else 0.0
    vmax_all = float(max(vmin_all + 1.0, edges["y_obs"].quantile(0.99))) if len(edges) else 1.0
    with open(SCALE_PATH, "w", encoding="utf-8") as fh:
        json.dump({"vmin": vmin_all, "vmax": vmax_all}, fh)

def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

g_edges = gpd.GeoDataFrame(edges, geometry=edges.apply(to_line, axis=1), crs="EPSG:4326")

# Plot
fig, ax = plt.subplots(figsize=(10, 9))
oas[~oas["is_test"]].plot(ax=ax, facecolor=ALL_FACE, edgecolor=ALL_EDGE, linewidth=0.6, zorder=1)
oas[oas["is_test"]].plot(ax=ax, facecolor=TEST_FACE, edgecolor=TEST_EDGE, linewidth=1.0, zorder=2)

if len(g_edges):
    norm = Normalize(vmin=vmin_all, vmax=vmax_all)
    cmap = get_cmap()
    g_plot = g_edges.sort_values("y_obs", ascending=True).reset_index(drop=True)
    vals = g_plot["y_obs"].clip(vmin_all, vmax_all).to_numpy()
    widths = 0.2 + 2.8 * norm(vals)
    colors = [cmap(norm(v)) for v in vals]
    for (_, r), lw, col in zip(g_plot.iterrows(), widths, colors):
        ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
                color=col, linewidth=lw, alpha=0.85, zorder=3)
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.set_title("Commuters\n(observado)", fontsize=9)

legend_elements = [
    Patch(facecolor=ALL_FACE, edgecolor=ALL_EDGE, label="Comunas (todas)"),
    Patch(facecolor=TEST_FACE, edgecolor=TEST_EDGE, label="Comunas en tiles TEST"),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False)
ax.set_title(
    f"Flujos observados — Provincia de Santiago (tiles TEST: {', '.join(test_tiles)})\n"
    "Todos los flujos intra-tile (sin filtrar)"
)
ax.set_axis_off()
plt.tight_layout()

png_path = IMGS / "observed_flows_TEST_all.png"
pdf_path = IMGS / "observed_flows_TEST_all.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print(f"[3.2] Escala compartida vmin={vmin_all:.0f}, vmax={vmax_all:.0f} | aristas={len(g_edges)}")

In [ ]:
# --- [GRÁFICO #4.1] Flujos predichos por DG + CPC_intra en el título ---

SCALE_PATH = IMGS / "yobs_color_scale.json"   # misma escala que 3.1/3.2

ALL_FACE, ALL_EDGE = "#f0f0f0", "#9e9e9e"
TEST_FACE, TEST_EDGE = "#ffd1dc", "#f48fb1"

# 1) Comunas y tiles TEST
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}

oas_test = set()
for t in test_tiles:
    oas_test |= set(map(str, tileid2oa.get(t, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# 2) Edges y coordenadas
edges_path = RES / "edges_TEST_pairs.csv"
edges = pd.read_csv(edges_path)
edges.columns = [c.strip() for c in edges.columns]
need_cols = {"origin", "destination", "y_obs", "y_pred"}
assert need_cols.issubset(edges.columns), f"{edges_path.name} debe tener columnas {need_cols}"

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].astype(str).map(get_lon)
    edges["lat_o"] = edges["origin"].astype(str).map(get_lat)
    edges["lon_d"] = edges["destination"].astype(str).map(get_lon)
    edges["lat_d"] = edges["destination"].astype(str).map(get_lat)

# 3) Filtro intra-tile
edges_f = edges[
    edges["origin"].astype(str).isin(oas_test) &
    edges["destination"].astype(str).isin(oas_test)
].copy()
same_tile = edges_f["origin"].astype(str).map(oa2tile) == edges_f["destination"].astype(str).map(oa2tile)
edges_f = edges_f[same_tile]
edges_f = edges_f[(edges_f["y_pred"] > 0) & (edges_f["y_obs"] >= 0)]
edges_f = edges_f.dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"])

# 4) CPC_intra
cpc_csv = RES / "tile2cpc_DG_chile.csv"
CPC_INTRA = None
if cpc_csv.exists():
    _df = pd.read_csv(cpc_csv)
    if "cpc_intra" in _df.columns:
        CPC_INTRA = float(_df["cpc_intra"].mean())
if CPC_INTRA is None:
    num = 2.0 * np.minimum(edges_f["y_pred"], edges_f["y_obs"]).sum()
    den = edges_f["y_pred"].sum() + edges_f["y_obs"].sum()
    CPC_INTRA = float(num / den) if den > 0 else float("nan")

# 5) Selección para visualizar
PRUNE_Q, PRUNE_ON = 0.75, "y_pred"
thr = edges_f[PRUNE_ON].quantile(PRUNE_Q) if len(edges_f) else 0.0
edges_plot = edges_f[edges_f[PRUNE_ON] >= thr].copy()
if edges_plot.empty and len(edges_f):
    edges_plot = edges_f.sort_values(PRUNE_ON, ascending=False).head(50).copy()
    thr = float(edges_plot[PRUNE_ON].min())

# 6) Escala compartida
if SCALE_PATH.exists():
    with open(SCALE_PATH, "r", encoding="utf-8") as fh:
        s = json.load(fh)
    vmin_all, vmax_all = float(s["vmin"]), float(s["vmax"])
else:
    vmin_all = float(edges["y_obs"].min()) if len(edges) else 0.0
    vmax_all = float(max(vmin_all + 1.0, edges["y_obs"].quantile(0.99))) if len(edges) else 1.0
    with open(SCALE_PATH, "w", encoding="utf-8") as fh:
        json.dump({"vmin": vmin_all, "vmax": vmax_all}, fh)

norm = Normalize(vmin=vmin_all, vmax=vmax_all)
cmap = get_cmap()

# 7) Geometrías
def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

g_edges = gpd.GeoDataFrame(edges_plot, geometry=edges_plot.apply(to_line, axis=1), crs="EPSG:4326")
g_plot = g_edges.sort_values("y_pred", ascending=True).reset_index(drop=True)
vals = g_plot["y_pred"].clip(vmin_all, vmax_all).to_numpy()
widths = 0.2 + 2.8 * norm(vals)
colors = [cmap(norm(v)) for v in vals]

# 8) Plot
fig, ax = plt.subplots(figsize=(10, 9))
oas[~oas["is_test"]].plot(ax=ax, facecolor=ALL_FACE, edgecolor=ALL_EDGE, linewidth=0.6, zorder=1)
oas[oas["is_test"]].plot(ax=ax, facecolor=TEST_FACE, edgecolor=TEST_EDGE, linewidth=1.0, zorder=2)

for (_, r), lw, col in zip(g_plot.iterrows(), widths, colors):
    ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
            color=col, linewidth=lw, alpha=0.85, zorder=3)

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.ax.set_title("Commuters\n(predicho)", fontsize=9)

legend_elements = [
    Patch(facecolor=ALL_FACE, edgecolor=ALL_EDGE, label="Comunas (todas)"),
    Patch(facecolor=TEST_FACE, edgecolor=TEST_EDGE, label="Comunas en tiles TEST"),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False)

tiles_tag = ", ".join(test_tiles)
cpc_tag = f"CPC_intra = {CPC_INTRA:.4f}" if np.isfinite(CPC_INTRA) else "CPC_intra no disponible"
thr_tag = f"Umbral p{int(PRUNE_Q*100)}: ≥ {thr:,.0f} commuters"
ax.set_title(f"DG (predicho) — Provincia de Santiago (tiles TEST: {tiles_tag})\n{cpc_tag} | {thr_tag}")
ax.set_axis_off()
plt.tight_layout()

png_path = IMGS / "predicted_flows_TEST.png"
pdf_path = IMGS / "predicted_flows_TEST.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print("Guardados:", png_path.name, "|", pdf_path.name)
print(f"Líneas dibujadas: {len(g_edges)} | Escala compartida vmin={vmin_all:.0f}, vmax={vmax_all:.0f} | Umbral={thr:,.0f}")

In [ ]:
# --- [GRÁFICO #4.2] Flujos predichos por DG (TODOS los intra-tile) ---

SCALE_PATH = IMGS / "yobs_color_scale.json"   # misma escala que 3.1/3.2

ALL_FACE, ALL_EDGE = "#f0f0f0", "#9e9e9e"
TEST_FACE, TEST_EDGE = "#ffd1dc", "#f48fb1"

# 1) Comunas y tiles TEST
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}

oas_test = set()
for t in test_tiles:
    oas_test |= set(map(str, tileid2oa.get(t, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# 2) Edges y coordenadas
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges.columns = [c.strip() for c in edges.columns]
assert {"origin", "destination", "y_obs", "y_pred"}.issubset(edges.columns)

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].astype(str).map(get_lon)
    edges["lat_o"] = edges["origin"].astype(str).map(get_lat)
    edges["lon_d"] = edges["destination"].astype(str).map(get_lon)
    edges["lat_d"] = edges["destination"].astype(str).map(get_lat)

# 3) Filtro intra-tile
edges_f = edges[
    edges["origin"].astype(str).isin(oas_test) &
    edges["destination"].astype(str).isin(oas_test)
].copy()
same_tile = edges_f["origin"].astype(str).map(oa2tile) == edges_f["destination"].astype(str).map(oa2tile)
edges_f = edges_f[same_tile].dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"])

# 4) CPC_intra para el título
CPC_INTRA = None
cpc_csv = RES / "tile2cpc_DG_chile.csv"
if cpc_csv.exists():
    _df = pd.read_csv(cpc_csv)
    if "cpc_intra" in _df.columns:
        CPC_INTRA = float(_df["cpc_intra"].mean())
if CPC_INTRA is None:
    num = 2.0 * np.minimum(edges_f["y_pred"], edges_f["y_obs"]).sum()
    den = edges_f["y_pred"].sum() + edges_f["y_obs"].sum()
    CPC_INTRA = float(num / den) if den > 0 else float("nan")

# 5) Escala compartida
if SCALE_PATH.exists():
    with open(SCALE_PATH, "r", encoding="utf-8") as fh:
        s = json.load(fh)
    vmin_all, vmax_all = float(s["vmin"]), float(s["vmax"])
else:
    vmin_all = float(edges["y_obs"].min()) if len(edges) else 0.0
    vmax_all = float(max(vmin_all + 1.0, edges["y_obs"].quantile(0.99))) if len(edges) else 1.0
    with open(SCALE_PATH, "w", encoding="utf-8") as fh:
        json.dump({"vmin": vmin_all, "vmax": vmax_all}, fh)

norm = Normalize(vmin=vmin_all, vmax=vmax_all)
cmap = get_cmap()

# 6) GeoDataFrame con TODAS las aristas intra-tile
def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

g_edges = gpd.GeoDataFrame(edges_f, geometry=edges_f.apply(to_line, axis=1), crs="EPSG:4326")
g_plot = g_edges.sort_values("y_pred", ascending=True).reset_index(drop=True)
vals = g_plot["y_pred"].clip(vmin_all, vmax_all).to_numpy()
widths = 0.2 + 2.8 * norm(vals)
colors = [cmap(norm(v)) for v in vals]

# 7) Plot
fig, ax = plt.subplots(figsize=(10, 9))
oas[~oas["is_test"]].plot(ax=ax, facecolor=ALL_FACE, edgecolor=ALL_EDGE, linewidth=0.6, zorder=1)
oas[oas["is_test"]].plot(ax=ax, facecolor=TEST_FACE, edgecolor=TEST_EDGE, linewidth=1.0, zorder=2)

for (_, r), lw, col in zip(g_plot.iterrows(), widths, colors):
    ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
            color=col, linewidth=lw, alpha=0.85, zorder=3)

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.ax.set_title("Commuters\n(predicho)", fontsize=9)

legend_elements = [
    Patch(facecolor=ALL_FACE, edgecolor=ALL_EDGE, label="Comunas (todas)"),
    Patch(facecolor=TEST_FACE, edgecolor=TEST_EDGE, label="Comunas en tiles TEST"),
]
ax.legend(handles=legend_elements, loc="lower left", frameon=False)

tiles_tag = ", ".join(test_tiles)
ax.set_title(
    f"DG (predicho) — Provincia de Santiago (tiles TEST: {tiles_tag})\n"
    "Todos los flujos intra-tile (sin filtrar)"
    + (f" | CPC_intra = {CPC_INTRA:.4f}" if np.isfinite(CPC_INTRA) else "")
)
ax.set_axis_off()
plt.tight_layout()

out_png = IMGS / "predicted_flows_TEST_all.png"
out_pdf = IMGS / "predicted_flows_TEST_all.pdf"
fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print("Guardados:", out_png.name, "|", out_pdf.name)
print(f"Aristas dibujadas: {len(g_plot)} | Escala compartida vmin={vmin_all:.0f}, vmax={vmax_all:.0f}")

In [ ]:
# === 5.1 [OBS vs PRED — SIDE BY SIDE] Flujos intra-tile (p75) ===

SCALE_PATH = IMGS / "yobs_color_scale.json"   # escala compartida basada en y_obs

ALL_FACE, ALL_EDGE = "#f0f0f0", "#9e9e9e"
TEST_FACE, TEST_EDGE = "#ffd1dc", "#f48fb1"

# Comunas y tiles TEST
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}

oas_test = set()
for t in test_tiles:
    oas_test |= set(map(str, tileid2oa.get(t, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# Edges desde evaluate()
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges.columns = [c.strip() for c in edges.columns]
assert {"origin", "destination", "y_obs", "y_pred"}.issubset(edges.columns)

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].astype(str).map(get_lon)
    edges["lat_o"] = edges["origin"].astype(str).map(get_lat)
    edges["lon_d"] = edges["destination"].astype(str).map(get_lon)
    edges["lat_d"] = edges["destination"].astype(str).map(get_lat)

# Filtro intra-tile
edges = edges[
    edges["origin"].astype(str).isin(oas_test) &
    edges["destination"].astype(str).isin(oas_test)
].copy()
same_tile = edges["origin"].astype(str).map(oa2tile) == edges["destination"].astype(str).map(oa2tile)
edges = edges[same_tile].dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"])

# Escala compartida
if SCALE_PATH.exists():
    s = json.loads(SCALE_PATH.read_text())
    vmin_all, vmax_all = float(s["vmin"]), float(s["vmax"])
else:
    vmin_all = float(edges["y_obs"].min()) if len(edges) else 0.0
    vmax_all = float(max(vmin_all + 1.0, edges["y_obs"].quantile(0.99))) if len(edges) else 1.0
    SCALE_PATH.write_text(json.dumps({"vmin": vmin_all, "vmax": vmax_all}, ensure_ascii=False))

norm, cmap = Normalize(vmin=vmin_all, vmax=vmax_all), get_cmap()

# Umbrales p75 independientes
PRUNE_Q = 0.75
thr_obs = edges["y_obs"].quantile(PRUNE_Q) if len(edges) else 0.0
thr_pred = edges["y_pred"].quantile(PRUNE_Q) if len(edges) else 0.0
edges_obs = edges[edges["y_obs"] >= thr_obs].copy()
edges_pred = edges[edges["y_pred"] >= thr_pred].copy()
if edges_obs.empty and len(edges):
    edges_obs = edges.nlargest(50, "y_obs").copy()
    thr_obs = float(edges_obs["y_obs"].min())
if edges_pred.empty and len(edges):
    edges_pred = edges.nlargest(50, "y_pred").copy()
    thr_pred = float(edges_pred["y_pred"].min())

# Helpers
def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

def draw_panel(ax, subset, value_col, subtitle):
    oas[~oas["is_test"]].plot(ax=ax, facecolor=ALL_FACE, edgecolor=ALL_EDGE, linewidth=0.6, zorder=1)
    oas[oas["is_test"]].plot(ax=ax, facecolor=TEST_FACE, edgecolor=TEST_EDGE, linewidth=1.0, zorder=2)
    if len(subset):
        g = gpd.GeoDataFrame(subset, geometry=subset.apply(to_line, axis=1), crs="EPSG:4326")
        g = g.sort_values(value_col, ascending=True).reset_index(drop=True)
        vals = g[value_col].clip(vmin_all, vmax_all).to_numpy()
        widths = 0.25 + 3.0 * norm(vals)
        colors = [cmap(norm(v)) for v in vals]
        for (_, r), lw, col in zip(g.iterrows(), widths, colors):
            ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
                    color=col, linewidth=lw, alpha=0.85, zorder=3)
    ax.set_axis_off()
    ax.set_title(subtitle, fontsize=11)

# Figura
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
draw_panel(axes[0], edges_obs, "y_obs", f"Observado — Umbral p75: ≥ {thr_obs:,.0f} commuters")
draw_panel(axes[1], edges_pred, "y_pred", f"Predicho — Umbral p75: ≥ {thr_pred:,.0f} commuters")

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.035, pad=0.02)
cbar.ax.set_title("Commuters\n(escala observada)", fontsize=9)

legend_elements = [
    Patch(facecolor=ALL_FACE, edgecolor=ALL_EDGE, label="Comunas (todas)"),
    Patch(facecolor=TEST_FACE, edgecolor=TEST_EDGE, label="Comunas en tiles TEST"),
]
axes[0].legend(handles=legend_elements, loc="lower left", frameon=False)

tiles_tag = ", ".join(test_tiles)
fig.suptitle(f"Flujos intra-tile — Provincia de Santiago (tiles TEST: {tiles_tag})", fontsize=13, y=0.98)
fig.tight_layout()

out_png = IMGS / "flows_obs_vs_pred_side_by_side.png"
out_pdf = IMGS / "flows_obs_vs_pred_side_by_side.pdf"
fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print("Guardados:", out_png.name, "|", out_pdf.name)
print(f"Escala observada: vmin={vmin_all:.0f}, vmax={vmax_all:.0f}")
print(f"Umbral obs p75={thr_obs:,.0f} | Umbral pred p75={thr_pred:,.0f}")
print(f"Aristas obs={len(edges_obs)} | pred={len(edges_pred)} (intra-tile)")

In [ ]:
# === 5.2 [OBS vs PRED — TODOS] Side-by-side de flujos intra-tile ===

SCALE_PATH = IMGS / "yobs_color_scale.json"   # misma escala que 3.*

ALL_FACE, ALL_EDGE = "#f0f0f0", "#9e9e9e"
TEST_FACE, TEST_EDGE = "#ffd1dc", "#f48fb1"

# Comunas y tiles TEST
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}

oas_test = set()
for t in test_tiles:
    oas_test |= set(map(str, tileid2oa.get(t, {}).keys()))
oas["is_test"] = oas["OA_ID"].isin(oas_test)

# Edges
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges.columns = [c.strip() for c in edges.columns]
assert {"origin", "destination", "y_obs", "y_pred"}.issubset(edges.columns)

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].astype(str).map(get_lon)
    edges["lat_o"] = edges["origin"].astype(str).map(get_lat)
    edges["lon_d"] = edges["destination"].astype(str).map(get_lon)
    edges["lat_d"] = edges["destination"].astype(str).map(get_lat)

# Filtro intra-tile
edges = edges[
    edges["origin"].astype(str).isin(oas_test) &
    edges["destination"].astype(str).isin(oas_test)
].copy()
same_tile = edges["origin"].astype(str).map(oa2tile) == edges["destination"].astype(str).map(oa2tile)
edges = edges[same_tile].dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"])

edges_obs = edges[edges["y_obs"] > 0].copy()
edges_pred = edges.copy()

# Escala compartida
if SCALE_PATH.exists():
    s = json.loads(SCALE_PATH.read_text())
    vmin_all, vmax_all = float(s["vmin"]), float(s["vmax"])
else:
    vmin_all = float(edges_obs["y_obs"].min()) if len(edges_obs) else 0.0
    vmax_all = float(max(vmin_all + 1.0, edges_obs["y_obs"].quantile(0.99))) if len(edges_obs) else 1.0
    SCALE_PATH.write_text(json.dumps({"vmin": vmin_all, "vmax": vmax_all}, ensure_ascii=False))

if vmin_all == vmax_all:
    vmax_all = vmin_all + 1.0

norm, cmap = Normalize(vmin=vmin_all, vmax=vmax_all), get_cmap()

# Helpers
def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

def draw_panel(ax, subset, value_col, subtitle):
    oas[~oas["is_test"]].plot(ax=ax, facecolor=ALL_FACE, edgecolor=ALL_EDGE, linewidth=0.6, zorder=1)
    oas[oas["is_test"]].plot(ax=ax, facecolor=TEST_FACE, edgecolor=TEST_EDGE, linewidth=1.0, zorder=2)
    if len(subset):
        g = gpd.GeoDataFrame(subset, geometry=subset.apply(to_line, axis=1), crs="EPSG:4326")
        g = g.sort_values(value_col, ascending=True).reset_index(drop=True)
        vals = g[value_col].clip(vmin_all, vmax_all).to_numpy()
        widths = 0.2 + 2.8 * norm(vals)
        colors = [cmap(norm(v)) for v in vals]
        for (_, r), lw, col in zip(g.iterrows(), widths, colors):
            ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
                    color=col, linewidth=lw, alpha=0.85, zorder=3)
    ax.set_axis_off()
    ax.set_title(subtitle, fontsize=11)

# Figura
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
draw_panel(axes[0], edges_obs, "y_obs", "Observado — Todos los flujos intra-tile")
draw_panel(axes[1], edges_pred, "y_pred", "Predicho — Todos los flujos intra-tile")

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.035, pad=0.02)
cbar.ax.set_title("Commuters\n(escala observada)", fontsize=9)

legend_elements = [
    Patch(facecolor=ALL_FACE, edgecolor=ALL_EDGE, label="Comunas (todas)"),
    Patch(facecolor=TEST_FACE, edgecolor=TEST_EDGE, label="Comunas en tiles TEST"),
]
axes[0].legend(handles=legend_elements, loc="lower left", frameon=False)

tiles_tag = ", ".join(test_tiles)
fig.suptitle(f"Flujos intra-tile — Provincia de Santiago (tiles TEST: {tiles_tag})", fontsize=13, y=0.98)
fig.tight_layout()

out_png = IMGS / "flows_obs_vs_pred_all_side_by_side.png"
out_pdf = IMGS / "flows_obs_vs_pred_all_side_by_side.pdf"
fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print("Guardados:", out_png.name, "|", out_pdf.name)
print(f"Escala observada: vmin={vmin_all:.0f}, vmax={vmax_all:.0f}")
print(f"Aristas obs={len(edges_obs)} | pred={len(edges_pred)} (intra-tile)")

# Análisis de interpretabilidad y comportamiento del modelo

## ¿Donde acierta y donde se equivoca?

In [ ]:
# === A1. Scatter y_obs vs y_pred (linear y log-log) ===

# Helper: formateo miles
fmt_int = FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", "."))

# Cargar tiles TEST y mapping OA->tile
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = set(oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys())

# Cargar edges de evaluate() y filtrar intra-tile en TEST
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

n_pairs = len(edges)
y_obs = edges["y_obs"].astype(float).to_numpy()
y_pred = edges["y_pred"].astype(float).to_numpy()

# Métricas globales
def safe_mape(y_true, y_hat, eps=1.0):
    denom = np.where(y_true < eps, eps, y_true)
    return np.mean(np.abs(y_hat - y_true) / denom)

if n_pairs > 0:
    SS_res = np.sum((y_obs - y_pred) ** 2)
    SS_tot = np.sum((y_obs - y_obs.mean()) ** 2)
    R2 = 1 - SS_res / SS_tot if SS_tot > 0 else np.nan
    r = np.corrcoef(y_obs, y_pred)[0, 1] if n_pairs > 1 else np.nan
else:
    R2 = np.nan
    r = np.nan

MAE = float(np.mean(np.abs(y_pred - y_obs))) if n_pairs > 0 else np.nan
RMSE = float(np.sqrt(np.mean((y_pred - y_obs) ** 2))) if n_pairs > 0 else np.nan
MAPE = float(safe_mape(y_obs, y_pred)) if n_pairs > 0 else np.nan

mask_hi = y_obs >= 50
MAPE_hi = float(safe_mape(y_obs[mask_hi], y_pred[mask_hi])) if mask_hi.sum() > 0 else np.nan

# CPC_intra recomputado
num = 2.0 * np.minimum(y_pred, y_obs).sum()
den = y_pred.sum() + y_obs.sum()
CPC = float(num / den) if den > 0 else np.nan

# Comparación con CSV de resultados
cpc_path = RES / "tile2cpc_DG_chile.csv"
CPC_file = np.nan
if cpc_path.exists():
    _cdf = pd.read_csv(cpc_path)
    if "cpc_intra" in _cdf.columns:
        CPC_file = float(_cdf["cpc_intra"].mean())
        print(f"CPC_intra (recalc)={CPC:0.4f} | CPC_intra (archivo)={CPC_file:0.4f} | Δ={CPC - CPC_file:0.4e}")
    else:
        print("Aviso: tile2cpc_DG_chile.csv existe pero no tiene columna 'cpc_intra'.")
else:
    print("Aviso: no se encontró tile2cpc_DG_chile.csv; solo CPC_intra recalculado.")

# R² en espacio log1p
mask_pos = (y_obs > 0) & (y_pred > 0)
if mask_pos.sum() > 1:
    log_obs = np.log1p(y_obs[mask_pos])
    log_pred = np.log1p(y_pred[mask_pos])
    SS_res_log = np.sum((log_obs - log_pred) ** 2)
    SS_tot_log = np.sum((log_obs - log_obs.mean()) ** 2)
    R2_log = 1 - SS_res_log / SS_tot_log if SS_tot_log > 0 else np.nan
    r_log = np.corrcoef(log_obs, log_pred)[0, 1]
else:
    R2_log = np.nan
    r_log = np.nan

print("\nA1 — Métricas globales (intra-tile, TEST)")
print(f"  pares                : {n_pairs}")
print(f"  sum y_obs            : {y_obs.sum():,.0f}")
print(f"  sum y_pred           : {y_pred.sum():,.0f}")
print(f"  CPC_intra (recalc)   : {CPC:0.4f}")
print(f"  R^2 (lineal)         : {R2:0.4f}")
print(f"  r (Pearson lineal)   : {r:0.4f}")
print(f"  R^2 (log1p)          : {R2_log:0.4f}")
print(f"  r (Pearson log1p)    : {r_log:0.4f}")
print(f"  MAE                  : {MAE:,.2f}")
print(f"  RMSE                 : {RMSE:,.2f}")
print(f"  MAPE global          : {MAPE:0.4f}")
print(f"  MAPE y_obs>=50       : {MAPE_hi:0.4f} (n={mask_hi.sum()})")

# Scatter (ejes lineales)
fig, ax = plt.subplots(figsize=(7.5, 7))
ax.scatter(y_obs, y_pred, s=14, alpha=0.5, edgecolor="none")
m = max(y_obs.max(), y_pred.max()) if n_pairs else 1.0
ax.plot([0, m], [0, m], color="#666666", linewidth=1.2, linestyle="--", label="y = x")
ax.set_xlabel("Flujo observado (commuters)")
ax.set_ylabel("Flujo predicho (commuters)")
ax.xaxis.set_major_formatter(fmt_int)
ax.yaxis.set_major_formatter(fmt_int)
ax.set_title(
    "DG Chile — Observado vs Predicho (intra-tile)\n"
    f"R²={R2:0.3f} | MAE={MAE:,.0f} | MAPE={MAPE:0.3f} | CPC={CPC:0.3f}"
)
ax.legend(frameon=False, loc="upper left")
ax.set_xlim(0, m * 1.02)
ax.set_ylim(0, m * 1.02)
plt.tight_layout()

p_linear = IMGS / "scatter_obs_pred_linear.png"
plt.savefig(p_linear, dpi=300, bbox_inches="tight")
plt.show()
print("Guardado:", p_linear.name)

# Scatter (log–log)
if mask_pos.sum() > 0:
    y_obs_pos = y_obs[mask_pos]
    y_pred_pos = y_pred[mask_pos]

    fig, ax = plt.subplots(figsize=(7.5, 7))
    ax.scatter(y_obs_pos, y_pred_pos, s=14, alpha=0.5, edgecolor="none")
    lo = y_obs_pos.min()
    hi = y_obs_pos.max()
    ax.plot([lo, hi], [lo, hi], color="#666666", linewidth=1.2, linestyle="--", label="y = x")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Flujo observado (log10)")
    ax.set_ylabel("Flujo predicho (log10)")
    ax.set_title(
        "DG Chile — Observado vs Predicho (log–log, intra-tile)\n"
        f"pares positivos={mask_pos.sum()} de {n_pairs} | R²_log1p={R2_log:0.3f}"
    )
    ax.legend(frameon=False, loc="upper left")
    plt.tight_layout()

    p_log = IMGS / "scatter_obs_pred_log.png"
    plt.savefig(p_log, dpi=300, bbox_inches="tight")
    plt.show()
    print("Guardado:", p_log.name)
else:
    print("No hay pares con y_obs>0 e y_pred>0 para el scatter log–log.")

In [ ]:
# === A2. Heatmaps OD (observado, predicho) + heatmap de error relativo ===

# Tiles TEST y mapping OA->tile
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = set(oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys())

# Edges de evaluate() y filtro intra-tile en TEST
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

# Tablas OD (OBS/PRED)
M_obs = edges.pivot_table(index="origin", columns="destination",
                          values="y_obs", aggfunc="sum", fill_value=0.0)
M_pred = edges.pivot_table(index="origin", columns="destination",
                           values="y_pred", aggfunc="sum", fill_value=0.0)

# Reindexar y ordenar por intensidad observada
row_strength = M_obs.sum(axis=1).sort_values(ascending=False).index
col_strength = M_obs.sum(axis=0).sort_values(ascending=False).index
M_obs = M_obs.reindex(index=row_strength, columns=col_strength, fill_value=0.0)
M_pred = M_pred.reindex(index=row_strength, columns=col_strength, fill_value=0.0)

# Error relativo estabilizado: (pred - obs) / (obs + 9)
denom_offset = 9.0
M_err = (M_pred - M_obs) / (M_obs + denom_offset)

# Guardar matrices para auditoría
M_obs.to_csv(RES / "matrix_obs.csv")
M_pred.to_csv(RES / "matrix_pred.csv")
M_err.to_csv(RES / "matrix_err.csv")

print(f"A2 — Matrices OD construidas. Shape: {M_obs.shape}. "
      f"sum_obs={M_obs.values.sum():,.0f} | sum_pred={M_pred.values.sum():,.0f}")

# Escala LOG compartida para OBS y PRED
pos_vals = M_obs.values[M_obs.values > 0]
if pos_vals.size:
    vmin_log = max(1.0, np.percentile(pos_vals, 1))
    vmax_log = np.percentile(pos_vals, 99)
else:
    vmin_log = vmax_log = 1.0
norm_log = LogNorm(vmin=vmin_log, vmax=vmax_log)
print(f"Escala log compartida OBS/PRED -> vmin={vmin_log:.1f}, vmax={vmax_log:.1f}")

def plot_heatmap(A, title, cmap="viridis", norm=None, fname="heatmap.png",
                 cbar_label="Flujo (commuters)"):
    n_rows, n_cols = A.shape
    fig, ax = plt.subplots(figsize=(8.8, 7.6))
    data = np.ma.masked_where(A.values == 0, A.values)
    im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.set_ylabel(cbar_label, rotation=90)
    ax.set_title(title)
    ax.set_xlabel("Destino (j)")
    ax.set_ylabel("Origen (i)")
    max_ticks = 20
    row_ticks = list(range(min(max_ticks, n_rows)))
    col_ticks = list(range(min(max_ticks, n_cols)))
    ax.set_yticks(row_ticks); ax.set_xticks(col_ticks)
    ax.set_yticklabels(list(A.index[:len(row_ticks)]), fontsize=7)
    ax.set_xticklabels(list(A.columns[:len(col_ticks)]), fontsize=7, rotation=90)
    plt.tight_layout()
    out = IMGS / fname
    plt.savefig(out, dpi=400, bbox_inches="tight", facecolor="white")
    plt.show()
    return out

p1 = plot_heatmap(
    M_obs,
    "Heatmap OD — Observado (intra-tile, TEST)\nordenado por outflow/inflow observados",
    cmap="viridis", norm=norm_log, fname="heatmap_obs.png",
    cbar_label="Flujo observado (commuters)"
)

p2 = plot_heatmap(
    M_pred,
    "Heatmap OD — Predicho por DG (intra-tile, TEST)\nmisma escala que observado",
    cmap="viridis", norm=norm_log, fname="heatmap_pred.png",
    cbar_label="Flujo predicho (commuters)"
)

finite_err = M_err.values[np.isfinite(M_err.values)]
lim = max(np.percentile(np.abs(finite_err), 95), 0.1) if finite_err.size else 1.0
norm_div = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)

p3 = plot_heatmap(
    M_err,
    "Heatmap OD — Error relativo ((pred−obs)/(obs+9))\nescala simétrica ±p95(|err|)",
    cmap="coolwarm", norm=norm_div, fname="heatmap_err.png",
    cbar_label="Error relativo"
)

print("Guardados:", p1.name, "|", p2.name, "|", p3.name)

In [ ]:
# === A3. Top errores (tabla) + mapa de errores ===

def haversine_km(lat1, lon1, lat2, lon2):
    """Distancia geodésica aproximada (km) entre dos puntos en EPSG:4326."""
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * asin(np.sqrt(a))

# Comunas (para mapa)
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

# OA->tile y set de TEST
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = set(oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys())

# Edges y coords
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].map(get_lon)
    edges["lat_o"] = edges["origin"].map(get_lat)
    edges["lon_d"] = edges["destination"].map(get_lon)
    edges["lat_d"] = edges["destination"].map(get_lat)

edges = edges.dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"]).copy()

# Métricas de error por arista
edges["y_obs"] = edges["y_obs"].astype(float)
edges["y_pred"] = edges["y_pred"].astype(float)

denom = edges["y_obs"] + 9.0
edges["err_rel"] = (edges["y_pred"] - edges["y_obs"]) / denom
edges["ape"] = edges["err_rel"].abs()
edges["abs_err"] = (edges["y_pred"] - edges["y_obs"]).abs()
edges["ratio"] = edges["y_pred"] / (edges["y_obs"] + 1e-9)

edges["dist_km"] = edges.apply(
    lambda r: haversine_km(r["lat_o"], r["lon_o"], r["lat_d"], r["lon_d"]),
    axis=1
)

edges["rank_obs_origin"] = edges.groupby("origin")["y_obs"].rank(ascending=False, method="min")
edges["rank_pred_origin"] = edges.groupby("origin")["y_pred"].rank(ascending=False, method="min")

# Guardar tabla completa
table_path = RES / "table_top_errors.csv"
edges.sort_values("ape", ascending=False).to_csv(table_path, index=False)
print("Tabla completa guardada:", table_path.name)

# Top over / under por error relativo
TOPN = 20
over_df = edges[edges["err_rel"] > 0].sort_values("ape", ascending=False).head(TOPN)
under_df = edges[edges["err_rel"] < 0].sort_values("ape", ascending=False).head(TOPN)

cols_show = [
    "origin", "destination",
    "y_obs", "y_pred", "abs_err", "err_rel", "ratio",
    "dist_km", "rank_obs_origin", "rank_pred_origin"
]
over_path = RES / "top_over.csv"
under_path = RES / "top_under.csv"
over_df[cols_show].to_csv(over_path, index=False)
under_df[cols_show].to_csv(under_path, index=False)
print("Top sobre-predichos:", over_path.name)
print("Top sub-predichos:", under_path.name)

# Mapa: Top-K por |err_rel|
TOPK = 50
top_edges = edges.sort_values("ape", ascending=False).head(TOPK).copy()

flow_scale = top_edges[["y_obs", "y_pred"]].max(axis=1)
w_min, w_max = 0.4, 3.2
if flow_scale.max() > 0:
    widths = w_min + (w_max - w_min) * (flow_scale - flow_scale.min()) / (flow_scale.max() - flow_scale.min() + 1e-9)
else:
    widths = np.full(len(top_edges), w_min)

lim = max(np.percentile(edges["err_rel"].abs(), 95), 0.1) if len(edges) else 1.0
norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
cmap = get_cmap("coolwarm")

def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

g_top = gpd.GeoDataFrame(top_edges, geometry=top_edges.apply(to_line, axis=1), crs="EPSG:4326")
g_top["_w"] = widths
g_top_sorted = g_top.sort_values("_w")

fig, ax = plt.subplots(figsize=(10, 9))
oas.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#9e9e9e", linewidth=0.6, zorder=1)
oas[oas["OA_ID"].isin(oas_test)].plot(ax=ax, facecolor="#ffd1dc", edgecolor="#f48fb1", linewidth=1.0, zorder=2)

for _, r in g_top_sorted.iterrows():
    ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
            color=cmap(norm(r["err_rel"])), linewidth=float(r["_w"]), alpha=0.9, zorder=3)

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.ax.set_title("Error relativo\n(pred−obs)/(obs+9)\n(azul=sub, rojo=sobre)", fontsize=8)

ax.set_title(f"Top-{TOPK} errores relativos | intra-tile (tiles TEST: {', '.join(test_tiles)})")
ax.set_axis_off()
plt.tight_layout()

png_path = IMGS / "map_edge_errors_topK.png"
pdf_path = IMGS / "map_edge_errors_topK.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print("Guardados:", png_path.name, "|", pdf_path.name)
print(f"TopK={TOPK} | escala simétrica ±{lim:0.2f}")

In [ ]:
# === A4. Mapa de errores (Top-K aristas intra-tile, TEST) ===

# Comunas (base del mapa)
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

# OA -> tile y OAs en tiles TEST
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = set(oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys())

# Edges
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].map(get_lon)
    edges["lat_o"] = edges["origin"].map(get_lat)
    edges["lon_d"] = edges["destination"].map(get_lon)
    edges["lat_d"] = edges["destination"].map(get_lat)

edges = edges.dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"]).copy()

# Métricas de error por arista
edges["y_obs"] = edges["y_obs"].astype(float)
edges["y_pred"] = edges["y_pred"].astype(float)
eps = 1e-9
edges["err_rel"] = (edges["y_pred"] - edges["y_obs"]) / (edges["y_obs"] + eps)
edges["ape"] = edges["err_rel"].abs()

# Descartar flujos muy pequeños para evitar APE exagerados
MIN_YOBS = 10
edges = edges[edges["y_obs"] >= MIN_YOBS].copy()

# Selección Top-K
TOPK = 50
top_edges = edges.sort_values("ape", ascending=False).head(TOPK).copy()

flow_scale = top_edges[["y_obs", "y_pred"]].max(axis=1)
w_min, w_max = 0.4, 3.2
if flow_scale.max() > flow_scale.min():
    widths = w_min + (w_max - w_min) * (flow_scale - flow_scale.min()) / (flow_scale.max() - flow_scale.min())
else:
    widths = np.full(len(top_edges), w_min)

lim = max(np.percentile(edges["ape"], 95), 0.1) if len(edges) else 1.0
norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
cmap = get_cmap("PRGn_r")   # negativo -> verde, positivo -> púrpura

def to_line(r):
    return LineString([(r["lon_o"], r["lat_o"]), (r["lon_d"], r["lat_d"])])

g_top = gpd.GeoDataFrame(
    top_edges.assign(_width=widths),
    geometry=top_edges.apply(to_line, axis=1),
    crs="EPSG:4326",
).sort_values("_width")

# Plot
fig, ax = plt.subplots(figsize=(10, 9))
oas.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#9e9e9e", linewidth=0.6, zorder=1)
oas[oas["OA_ID"].isin(oas_test)].plot(ax=ax, facecolor="#ffd1dc", edgecolor="#f48fb1", linewidth=1.0, zorder=2)

for _, r in g_top.iterrows():
    ax.plot([r["lon_o"], r["lon_d"]], [r["lat_o"], r["lat_d"]],
            color=cmap(norm(r["err_rel"])), linewidth=float(r["_width"]), alpha=0.9, zorder=3)

sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
cbar.ax.set_title("Error relativo\n(pred−obs)/(obs+ε)\nverde=sub | púrpura=sobre", fontsize=8)

ax.set_title(f"Top-{TOPK} errores relativos | intra-tile (tiles TEST: {', '.join(test_tiles)})")
ax.set_axis_off()
plt.tight_layout()

png_path = IMGS / "map_edge_errors_topK_A4.png"
pdf_path = IMGS / "map_edge_errors_topK_A4.pdf"
plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.1, facecolor="white")
plt.show()

print("Guardados:", png_path.name, "|", pdf_path.name)
print(f"TopK={TOPK} | escala simétrica ±{lim:0.2f} (error relativo)")

In [ ]:
# === A5. Rendimiento por origen: CPC_i y MAPE_i (intra-tile, TEST) ===

# 1. Capas base
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = {oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys()}

# 2. Edges intra-tile TEST
edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges["y_obs"] = edges["y_obs"].astype(float)
edges["y_pred"] = edges["y_pred"].astype(float)

edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

# 3. Métricas por origen
def metrics_per_origin(df):
    y_o = df["y_obs"].to_numpy(dtype=float)
    y_p = df["y_pred"].to_numpy(dtype=float)
    num = 2.0 * np.minimum(y_o, y_p).sum()
    den = y_o.sum() + y_p.sum()
    cpc_i = float(num / den) if den > 0 else np.nan
    ape_eps = np.abs(y_p - y_o) / (y_o + 9.0)
    mape_i = float(ape_eps.mean()) if len(ape_eps) else np.nan
    return pd.Series({
        "CPC_i": cpc_i,
        "MAPE_i": mape_i,
        "O_obs_intra": y_o.sum(),
        "O_pred_intra": y_p.sum(),
        "n_dests": len(df)
    })

origin_perf = (
    edges.groupby("origin")
         .apply(metrics_per_origin)
         .reset_index()
         .rename(columns={"origin": "OA_ID"})
)

y_obs_all = edges["y_obs"].to_numpy(dtype=float)
y_pred_all = edges["y_pred"].to_numpy(dtype=float)
num_global = 2.0 * np.minimum(y_obs_all, y_pred_all).sum()
den_global = y_obs_all.sum() + y_pred_all.sum()
CPC_global = float(num_global / den_global) if den_global > 0 else np.nan

perf_path = RES / "origin_performance_metrics.csv"
origin_perf.to_csv(perf_path, index=False)
print("A5 — métricas por origen guardadas en:", perf_path.name)
print("  Orígenes en TEST:", len(origin_perf))
print("  CPC_intra global (referencia):", f"{CPC_global:.4f}")
print("  CPC_i rango: [{:.4f}, {:.4f}]".format(origin_perf["CPC_i"].min(), origin_perf["CPC_i"].max()))
print("  MAPE_i rango: [{:.3f}, {:.3f}]".format(origin_perf["MAPE_i"].min(), origin_perf["MAPE_i"].max()))

# 4. Gráfico de barras CPC_i por OA
perf_sorted = origin_perf.sort_values("CPC_i", ascending=False).reset_index(drop=True)
x = np.arange(len(perf_sorted))
cpc_vals = perf_sorted["CPC_i"].to_numpy()

norm_cpc = Normalize(vmin=np.nanmin(cpc_vals), vmax=np.nanmax(cpc_vals))
cmap_cpc = get_cmap("viridis")
colors = cmap_cpc(norm_cpc(cpc_vals))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, cpc_vals, color=colors, edgecolor="black", linewidth=0.4)
ax.axhline(CPC_global, color="#444444", linestyle="--", linewidth=1.0,
           label=f"CPC_intra global = {CPC_global:.3f}")
ax.set_xticks(x)
ax.set_xticklabels(perf_sorted["OA_ID"], rotation=90, fontsize=8)
ax.set_ylabel("CPC_i (intra-tile)")
ax.set_xlabel("Comuna de origen (OA_ID)")
ax.set_title("DeepGravity Chile — CPC_i por comuna de origen (tiles TEST)")

sm = ScalarMappable(norm=norm_cpc, cmap=cmap_cpc); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.ax.set_ylabel("CPC_i", rotation=90)
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()

bar_path_png = IMGS / "bar_cpc_by_origin.png"
bar_path_pdf = IMGS / "bar_cpc_by_origin.pdf"
fig.savefig(bar_path_png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(bar_path_pdf, bbox_inches="tight", facecolor="white")
plt.show()
print("Gráficos de barras guardados:", bar_path_png.name, "|", bar_path_pdf.name)

# 5. Choropleth de MAPE_i por comuna
oas_perf = oas.merge(origin_perf, on="OA_ID", how="left")
oas_perf["is_test"] = oas_perf["OA_ID"].isin(origin_perf["OA_ID"])

mape_vals = oas_perf.loc[oas_perf["is_test"], "MAPE_i"].dropna().to_numpy()
if mape_vals.size:
    vmax_mape = float(max(np.percentile(mape_vals, 95), mape_vals.min() + 1e-3))
    vmin_mape = 0.0
else:
    vmin_mape, vmax_mape = 0.0, 1.0

norm_mape = Normalize(vmin=vmin_mape, vmax=vmax_mape)
cmap_mape = get_cmap("magma_r")

fig, ax = plt.subplots(figsize=(10, 9))
oas_perf.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#9e9e9e", linewidth=0.6, zorder=1)

oas_test_perf = oas_perf[oas_perf["is_test"]].copy()
if not oas_test_perf.empty:
    oas_test_perf["MAPE_clip"] = oas_test_perf["MAPE_i"].clip(vmin_mape, vmax_mape)
    oas_test_perf.plot(column="MAPE_clip", cmap=cmap_mape, norm=norm_mape,
                       ax=ax, linewidth=1.0, edgecolor="#555555", zorder=2, legend=False)
    sm2 = ScalarMappable(norm=norm_mape, cmap=cmap_mape); sm2.set_array([])
    cbar = plt.colorbar(sm2, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.set_title("MAPE_i\n|pred−obs|/(obs+9)", fontsize=9)

ax.set_title("DeepGravity Chile — MAPE_i por comuna de origen\n(intra-tile, tiles TEST)")
ax.set_axis_off()
plt.tight_layout()

map_path_png = IMGS / "map_mape_by_origin.png"
map_path_pdf = IMGS / "map_mape_by_origin.pdf"
fig.savefig(map_path_png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(map_path_pdf, bbox_inches="tight", facecolor="white")
plt.show()
print("Mapas guardados:", map_path_png.name, "|", map_path_pdf.name)

In [ ]:
# === A6. Top-K recall por origen (intra-tile, TEST) ===

# 1. Capas y filtros base
oas = gpd.read_file(cfg.OUTPUT_AREAS_SHP, engine="pyogrio")
if oas.crs is None or oas.crs.to_epsg() != 4326:
    oas = oas.to_crs(4326)
oas["OA_ID"] = oas["OA_ID"].astype(str)

with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = {oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys()}

edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges["y_obs"] = edges["y_obs"].astype(float)
edges["y_pred"] = edges["y_pred"].astype(float)

edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

print(f"A6 — edges intra-tile en TEST: {len(edges)} filas, {edges['origin'].nunique()} orígenes")

# 2. recall@K por origen (K=1,3,5)
Ks = [1, 3, 5]
rows = []

for ori, df_ori in edges.groupby("origin"):
    df_ori = df_ori.copy()
    order_obs = df_ori.sort_values("y_obs", ascending=False)["destination"].tolist()
    order_pred = df_ori.sort_values("y_pred", ascending=False)["destination"].tolist()

    rec_vals = {}
    for K in Ks:
        inter = len(set(order_obs[:K]) & set(order_pred[:K]))
        rec_vals[f"recall_at_{K}"] = inter / float(K)

    row = {
        "OA_ID": ori,
        "n_dests": len(df_ori),
        "n_obs_pos": int((df_ori["y_obs"] > 0).sum()),
        "O_obs_intra": float(df_ori["y_obs"].sum()),
        "O_pred_intra": float(df_ori["y_pred"].sum()),
    }
    row.update(rec_vals)
    rows.append(row)

recall_df = pd.DataFrame(rows).sort_values("OA_ID").reset_index(drop=True)

recall_path = RES / "recall_at_k_by_origin.csv"
recall_df.to_csv(recall_path, index=False)
print("Tabla recall@K por origen guardada en:", recall_path.name)

print("\nResumen recall@K (intra-tile, TEST)")
print(f"  n_orígenes = {len(recall_df)}")
for K in Ks:
    vals = recall_df[f"recall_at_{K}"].to_numpy()
    print(f"  K={K}: mean={vals.mean():.3f}, median={np.median(vals):.3f}, "
          f"min={vals.min():.3f}, max={vals.max():.3f}")

# 3. Boxplot de distribución recall@K
fig, ax = plt.subplots(figsize=(7, 5))
data_box = [recall_df[f"recall_at_{K}"].to_numpy() for K in Ks]
ax.boxplot(data_box, positions=range(1, len(Ks)+1), widths=0.6)
ax.set_xticks(range(1, len(Ks)+1))
ax.set_xticklabels([f"K={K}" for K in Ks])
ax.set_ylabel("recall@K")
ax.set_ylim(-0.05, 1.05)
ax.set_title("DeepGravity Chile — Distribución de recall@K por comuna de origen\n(intra-tile, tiles TEST)")
plt.tight_layout()

box_png = IMGS / "recall_at_k_boxplot.png"
box_pdf = IMGS / "recall_at_k_boxplot.pdf"
fig.savefig(box_png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(box_pdf, bbox_inches="tight", facecolor="white")
plt.show()
print("Boxplot guardado:", box_png.name, "|", box_pdf.name)

# 4. Choropleth de recall@K por comuna
K_map = 5
col_map = f"recall_at_{K_map}"

oas_recall = oas.merge(recall_df, on="OA_ID", how="left")
oas_recall["is_test"] = oas_recall["OA_ID"].isin(recall_df["OA_ID"])

fig, ax = plt.subplots(figsize=(10, 9))
oas_recall.plot(ax=ax, facecolor="#f0f0f0", edgecolor="#9e9e9e", linewidth=0.6, zorder=1)

oas_test_recall = oas_recall[oas_recall["is_test"]].copy()
if not oas_test_recall.empty:
    norm = Normalize(vmin=0.0, vmax=1.0)
    cmap = get_cmap("YlGnBu")
    oas_test_recall.plot(column=col_map, cmap=cmap, norm=norm,
                         ax=ax, linewidth=1.0, edgecolor="#555555", zorder=2, legend=False)
    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
    cbar.ax.set_title(f"recall@{K_map}", fontsize=9)

ax.set_title(
    f"DeepGravity Chile — recall@{K_map} por comuna de origen\n"
    f"(intra-tile, tiles TEST: {', '.join(test_tiles)})"
)
ax.set_axis_off()
plt.tight_layout()

map_png = IMGS / f"map_recall_at{K_map}_by_origin.png"
map_pdf = IMGS / f"map_recall_at{K_map}_by_origin.pdf"
fig.savefig(map_png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(map_pdf, bbox_inches="tight", facecolor="white")
plt.show()
print("Mapas guardados:", map_png.name, "|", map_pdf.name)

In [ ]:
# === A7. Sesgo vs distancia (bins por deciles de d_ij) ===

def haversine_km(lat1, lon1, lat2, lon2):
    """Distancia geodésica aproximada (km) entre dos puntos (lat, lon) en EPSG:4326."""
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * asin(np.sqrt(a))

# 1. Tiles TEST y edges intra-tile
with open(PROCESSED / "tileid2oa2handmade_features.json", "r", encoding="utf-8") as fh:
    tileid2oa = json.load(fh)
test_tiles = pd.read_csv(PROCESSED / "test_tiles.csv", header=None, dtype=str)[0].tolist()
oa2tile = {str(oa): str(tid) for tid, d in tileid2oa.items() for oa in d.keys()}
oas_test = {oa for t in test_tiles for oa in tileid2oa.get(t, {}).keys()}

edges = pd.read_csv(RES / "edges_TEST_pairs.csv")
edges["origin"] = edges["origin"].astype(str)
edges["destination"] = edges["destination"].astype(str)
edges["y_obs"] = edges["y_obs"].astype(float)
edges["y_pred"] = edges["y_pred"].astype(float)

edges = edges[
    edges["origin"].isin(oas_test) & edges["destination"].isin(oas_test)
].copy()
same_tile = edges["origin"].map(oa2tile) == edges["destination"].map(oa2tile)
edges = edges[same_tile].copy()

print(f"A7 — edges intra-tile en TEST: {len(edges)} filas, {edges['origin'].nunique()} orígenes")

# 2. Distancias d_ij (km)
if not {"lon_o", "lat_o", "lon_d", "lat_d"}.issubset(edges.columns):
    with open(PROCESSED / "oa2centroid.pkl", "rb") as fh:
        oa2centroid = pickle.load(fh)

    def get_lon(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[1])

    def get_lat(oa):
        v = oa2centroid.get(str(oa)); return np.nan if v is None else float(v[0])

    edges["lon_o"] = edges["origin"].map(get_lon)
    edges["lat_o"] = edges["origin"].map(get_lat)
    edges["lon_d"] = edges["destination"].map(get_lon)
    edges["lat_d"] = edges["destination"].map(get_lat)

edges = edges.dropna(subset=["lon_o", "lat_o", "lon_d", "lat_d"]).copy()

if "dist_km" not in edges.columns:
    edges["dist_km"] = edges.apply(
        lambda r: haversine_km(r["lat_o"], r["lon_o"], r["lat_d"], r["lon_d"]),
        axis=1
    )

# 3. Error relativo y APE
denom = edges["y_obs"] + 9.0
edges["err_rel"] = (edges["y_pred"] - edges["y_obs"]) / denom
edges["ape"] = edges["err_rel"].abs()

# 4. Bins por deciles de distancia
edges = edges.sort_values("dist_km").copy()
edges["dist_bin"] = pd.qcut(edges["dist_km"], q=10, duplicates="drop")

stats = []
for cat, g in edges.groupby("dist_bin", observed=True):
    if g.empty:
        continue
    dist_vals = g["dist_km"].to_numpy(dtype=float)
    err_vals = g["err_rel"].to_numpy(dtype=float)
    ape_vals = g["ape"].to_numpy(dtype=float)
    stats.append({
        "bin": str(cat),
        "dist_min": float(dist_vals.min()),
        "dist_max": float(dist_vals.max()),
        "dist_mid": float(dist_vals.mean()),
        "n_pairs": int(len(g)),
        "mean_err_rel": float(err_vals.mean()),
        "std_err_rel": float(err_vals.std(ddof=1) / np.sqrt(len(err_vals))) if len(err_vals) > 1 else 0.0,
        "median_APE": float(np.median(ape_vals)),
    })

bias_df = pd.DataFrame(stats).sort_values("dist_mid").reset_index(drop=True)

bias_path = RES / "bias_vs_distance.csv"
bias_df.to_csv(bias_path, index=False)
print("A7 — tabla bias_vs_distance guardada en:", bias_path.name)
print(bias_df[["dist_mid", "n_pairs", "mean_err_rel", "median_APE"]])

# 5. Gráfico línea + bandas
x = bias_df["dist_mid"].to_numpy()
y_mean = bias_df["mean_err_rel"].to_numpy()
y_lo = y_mean - bias_df["std_err_rel"].to_numpy()
y_hi = y_mean + bias_df["std_err_rel"].to_numpy()
y_ape = bias_df["median_APE"].to_numpy()

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(x, y_mean, marker="o", linestyle="-", color="tab:blue", label="Error relativo medio")
ax1.fill_between(x, y_lo, y_hi, color="tab:blue", alpha=0.2, label="±1·SE error relativo")
ax1.axhline(0.0, color="#666666", linestyle="--", linewidth=1.0)
ax1.set_xlabel("Distancia OD (km)")
ax1.set_ylabel("Error relativo medio\n(pred−obs)/(obs+9)")
ax1.set_title("DeepGravity Chile — Sesgo vs distancia\n(intra-tile, tiles TEST)")
ax1.grid(alpha=0.2)

ax2 = ax1.twinx()
ax2.plot(x, y_ape, marker="s", linestyle="--", color="tab:orange", label="Mediana APE")
ax2.set_ylabel("Mediana APE\n|pred−obs|/(obs+9)")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False)
plt.tight_layout()

png_path = IMGS / "bias_vs_distance.png"
pdf_path = IMGS / "bias_vs_distance.pdf"
fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.show()

print("Gráficos guardados:", png_path.name, "|", pdf_path.name)

## ¿Por qué se equivoca? (atribución de features)

In [ ]:
# === Análisis de impacto de features en el rendimiento por origen ===

feat_path = cfg.FEATURES_CSV
perf_path = RES / "origin_performance_metrics.csv"
print("Insumos: features.csv y results/origin_performance_metrics.csv")

# 1. Carga y merge OA_ID <-> features
features = pd.read_csv(feat_path)
perf = pd.read_csv(perf_path)

print("\nCols features:", list(features.columns))
print("Cols perf    :", list(perf.columns))

df = perf.merge(features, on="OA_ID", how="left")
print("\nMerged shape:", df.shape)

# 2. Features derivados básicos
df["density"] = df["population"] / df["area_km2"].replace(0, np.nan)
df["log_pop"] = np.log1p(df["population"])
df["log_density"] = np.log1p(df["density"])

# 3. Correlaciones Pearson CPC_i / MAPE_i con features numéricas
perf_cols = ["CPC_i", "MAPE_i", "O_obs_intra", "O_pred_intra", "n_dests"]
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in perf_cols]

corr_cpc = []
corr_mape = []
for f in feature_cols:
    x = df[f]
    if x.isna().all():
        continue
    corr_cpc.append({"feature": f, "pearson_CPC_i": x.corr(df["CPC_i"])})
    corr_mape.append({"feature": f, "pearson_MAPE_i": x.corr(df["MAPE_i"])})

corr_cpc_df = pd.DataFrame(corr_cpc).dropna().sort_values(
    "pearson_CPC_i", key=lambda s: s.abs(), ascending=False
)
corr_mape_df = pd.DataFrame(corr_mape).dropna().sort_values(
    "pearson_MAPE_i", key=lambda s: s.abs(), ascending=False
)

corr_cpc_path = RES / "feature_correlations_CPC_i.csv"
corr_mape_path = RES / "feature_correlations_MAPE_i.csv"
corr_cpc_df.to_csv(corr_cpc_path, index=False)
corr_mape_df.to_csv(corr_mape_path, index=False)

print("\nCorrelaciones con CPC_i (ordenadas por |r|):")
print(corr_cpc_df.to_string(index=False, float_format=lambda v: f"{v: .3f}"))
print("\nCorrelaciones con MAPE_i (ordenadas por |r|):")
print(corr_mape_df.to_string(index=False, float_format=lambda v: f"{v: .3f}"))
print("\nGuardado:", corr_cpc_path.name, "|", corr_mape_path.name)

# 4. Scatterplots para features clave
def scatter_perf_vs_feature(df, feat_col, y_col, fname, log_x=False):
    if feat_col not in df.columns:
        print(f"[Aviso] {feat_col} no está en el DataFrame, no se genera {fname}")
        return
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(df[feat_col], df[y_col], s=40, alpha=0.8)
    for _, r in df.iterrows():
        ax.annotate(str(r["OA_ID"]), (r[feat_col], r[y_col]),
                    textcoords="offset points", xytext=(3, 3), fontsize=7)
    if log_x:
        ax.set_xscale("log")
    ax.set_xlabel(feat_col)
    ax.set_ylabel(y_col)
    ax.set_title(f"{y_col} vs {feat_col}")
    plt.tight_layout()
    out = IMGS / fname
    plt.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Guardado scatter:", out.name)

scatter_perf_vs_feature(df, "population", "CPC_i", "scatter_CPCi_vs_population.png", log_x=True)
scatter_perf_vs_feature(df, "population", "MAPE_i", "scatter_MAPEi_vs_population.png", log_x=True)
scatter_perf_vs_feature(df, "density", "MAPE_i", "scatter_MAPEi_vs_density.png", log_x=True)
scatter_perf_vs_feature(df, "retail_pois", "MAPE_i", "scatter_MAPEi_vs_retail_pois.png", log_x=False)

print("\nAnálisis de correlaciones terminado.")

## Alcance de este cuaderno

Los Chunks 0 a 4 producen los objetos serializados que Deep Gravity consume
desde `data/chile/processed/`: centroides, mapeo tesela–unidad, partición
espacial y vector de atributos por unidad. Todos derivan de infraestructura
territorial pública y forman parte de la entrega del repositorio.

El Chunk 5 y las secciones de visualización y análisis documentan el
procedimiento de construcción de la variable objetivo y de auditoría de la
corrida. Sus insumos no se distribuyen; el código se publica para que el
procedimiento sea examinable y reejecutable sobre una fuente equivalente.